# QC: Фин. рез. по месяцам — Excel vs `final_df`

Самостоятельная сверка, без полного прогона `final_script_2`.

**Lake:** `SUM(fin_result)` из CSV / checkpoint / `final_df` в памяти.  
**Excel:** колонка `Фин. Рез.` / `Фин.рез.` / `Fin.Res.` в месячных файлах `01_Январь_2026.xlsx` …

По умолчанию Excel в конфиге — **янв–июнь**. Июль/август подхватятся сами, если файлы лежат в `DATA_DIR`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')

excel_header = 0
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
}

excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
}

for extra_month, extra_name in [
    ('2026-07', '07_Июль_2026.xlsx'),
    ('2026-08', '08_Август_2026.xlsx'),
]:
    extra_path = DATA_DIR / extra_name
    if extra_path.exists():
        excel_reference_by_month[extra_month] = extra_path

final_df_candidates = [
    DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_08_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv',
]
checkpoint_dirs = [
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_08_final_script_2',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_07_final_script_2',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_06_mpos',
]

FIN_RESULT_EXCEL_COLS = [
    'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
    'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
    'fin_result', 'Fin.Res.', 'FinRes',
]


def month_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if len(s) >= 7 and s[4] == '-':
        return s[:7]
    try:
        return pd.to_datetime(s).strftime('%Y-%m')
    except Exception:
        return None


def to_num_series(s):
    if s is None:
        return pd.Series(dtype='float64')
    if not isinstance(s, pd.Series):
        s = pd.Series(s)
    raw = s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False)
    raw = raw.str.replace(',', '.', regex=False)
    return pd.to_numeric(raw, errors='coerce')


def pick_col(columns, candidates):
    norm = {str(c).strip(): c for c in columns}
    for name in candidates:
        if name in norm:
            return norm[name]
    lower = {str(c).strip().lower().replace(' ', ''): c for c in columns}
    for name in candidates:
        key = name.strip().lower().replace(' ', '')
        if key in lower:
            return lower[key]
    return None


print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
for k, v in excel_reference_by_month.items():
    print(f'Excel {k}: exists={Path(v).exists()} | {v}')


In [ ]:
# 1) final_df: память → CSV → checkpoints
final_df = None
source_used = None

if 'final_df_period_df' in globals() and final_df_period_df is not None and len(final_df_period_df):
    final_df = final_df_period_df.copy()
    source_used = 'final_df_period_df (memory)'
elif 'final_df' in globals() and globals().get('final_df') is not None and len(globals()['final_df']):
    final_df = globals()['final_df'].copy()
    source_used = 'final_df (memory)'
else:
    for p in final_df_candidates:
        if p.exists():
            final_df = pd.read_csv(p, dtype=str, low_memory=False)
            source_used = f'csv: {p}'
            break

if final_df is None:
    frames = []
    used_dir = None
    for d in checkpoint_dirs:
        if not d.exists():
            continue
        month_files = sorted(d.glob('final_df_2026_*.parquet')) + sorted(d.glob('final_df_2026_*.csv'))
        if not month_files:
            continue
        used_dir = d
        seen = set()
        for f in month_files:
            key = f.stem
            if key in seen:
                continue
            seen.add(key)
            if f.suffix == '.parquet':
                frames.append(pd.read_parquet(f))
            else:
                frames.append(pd.read_csv(f, dtype=str, low_memory=False))
        break
    if frames:
        final_df = pd.concat(frames, ignore_index=True)
        source_used = f'checkpoints: {used_dir}'

if final_df is None or final_df.empty:
    raise RuntimeError(
        'final_df не найден. Положите CSV в DATA_DIR или откройте эту тетрадку '
        'после period-ячейки final_script_2.'
    )

need = {'report_month', 'fin_result'}
missing = need - set(final_df.columns)
if missing:
    raise RuntimeError(f'В final_df нет колонок: {missing}. Есть: {list(final_df.columns)}')

work = final_df.copy()
work['report_month'] = work['report_month'].map(month_key)
work['fin_result'] = to_num_series(work['fin_result'])
for extra in ['chod', 'aur', 'amortization']:
    if extra in work.columns:
        work[extra] = to_num_series(work[extra])

print('source:', source_used)
print('rows:', f'{len(work):,}', '| months:', sorted(work['report_month'].dropna().unique().tolist()))
display(work.groupby('report_month', as_index=False)['fin_result'].sum())


In [ ]:
# 2) Excel: SUM(Фин. Рез.) по месяцам
excel_rows = []
for month, path in excel_reference_by_month.items():
    path = Path(path)
    if not path.exists():
        excel_rows.append({
            'report_month': month,
            'fin_result_excel': np.nan,
            'excel_col': None,
            'excel_rows': 0,
            'reference_status': 'file_missing',
            'excel_path': str(path),
        })
        continue
    header = excel_header_by_month.get(month, excel_header)
    ex = pd.read_excel(path, header=header)
    col = pick_col(ex.columns, FIN_RESULT_EXCEL_COLS)
    if col is None:
        excel_rows.append({
            'report_month': month,
            'fin_result_excel': np.nan,
            'excel_col': None,
            'excel_rows': int(len(ex)),
            'reference_status': 'column_missing',
            'excel_path': str(path),
        })
        print(f'{month}: нет колонки Фин. Рез. Колонки: {list(ex.columns)}')
        continue
    val = float(to_num_series(ex[col]).fillna(0).sum())
    excel_rows.append({
        'report_month': month,
        'fin_result_excel': val,
        'excel_col': col,
        'excel_rows': int(len(ex)),
        'reference_status': 'has_excel_reference',
        'excel_path': str(path),
    })

excel_month = pd.DataFrame(excel_rows)
print('Excel months:')
display(excel_month)


In [ ]:
# 3) Сводка lake vs Excel
agg = {'fin_result': 'sum'}
for extra in ['chod', 'aur', 'amortization']:
    if extra in work.columns:
        agg[extra] = 'sum'

lake_month = work.groupby('report_month', as_index=False).agg(agg)
lake_month = lake_month.rename(columns={
    'fin_result': 'fin_result_lake',
    'chod': 'chod_lake',
    'aur': 'aur_lake',
    'amortization': 'amortization_lake',
})

cmp = lake_month.merge(excel_month, on='report_month', how='outer')
cmp['delta_lake_minus_excel'] = cmp['fin_result_lake'] - cmp['fin_result_excel']
cmp['delta_pct_vs_excel'] = np.where(
    cmp['fin_result_excel'].abs() > 1e-9,
    100.0 * cmp['delta_lake_minus_excel'] / cmp['fin_result_excel'],
    np.nan,
)
if {'chod_lake', 'aur_lake', 'amortization_lake'}.issubset(cmp.columns):
    cmp['fin_result_from_parts'] = (
        cmp['chod_lake'].fillna(0) - cmp['aur_lake'].fillna(0) - cmp['amortization_lake'].fillna(0)
    )
    cmp['parts_minus_fin_result'] = cmp['fin_result_from_parts'] - cmp['fin_result_lake'].fillna(0)

cmp = cmp.sort_values('report_month').reset_index(drop=True)

print('=== Фин. рез. по месяцам: final_df vs Excel ===')
display(cmp[[
    'report_month', 'fin_result_lake', 'fin_result_excel',
    'delta_lake_minus_excel', 'delta_pct_vs_excel', 'reference_status',
]])

with_xl = cmp.loc[cmp['reference_status'] == 'has_excel_reference']
sum_lake_xl = float(with_xl['fin_result_lake'].fillna(0).sum()) if len(with_xl) else 0.0
sum_excel = float(with_xl['fin_result_excel'].fillna(0).sum()) if len(with_xl) else 0.0
sum_delta = sum_lake_xl - sum_excel
sum_delta_pct = (sum_delta / sum_excel * 100.0) if abs(sum_excel) > 1e-9 else np.nan

print('=== Итого по месяцам с Excel ===')
print(f'  fin_result_lake  = {sum_lake_xl:,.2f}')
print(f'  fin_result_excel = {sum_excel:,.2f}')
print(f'  delta (lake-excel) = {sum_delta:,.2f}')
print(f'  delta_pct = {sum_delta_pct:,.2f}%' if pd.notna(sum_delta_pct) else '  delta_pct = n/a')
print(f'  lake все месяцы = {float(cmp["fin_result_lake"].fillna(0).sum()):,.2f}')

if {'chod_lake', 'aur_lake', 'amortization_lake'}.issubset(cmp.columns):
    print('=== Компоненты lake: chod − aur − amort ≈ fin_result ===')
    display(cmp[[
        'report_month', 'fin_result_lake', 'fin_result_from_parts', 'parts_minus_fin_result',
        'chod_lake', 'aur_lake', 'amortization_lake',
    ]])

out_csv = DATA_DIR / 'fin_result_lake_vs_excel_by_month.csv'
cmp.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('Saved:', out_csv)
